# Preparación de datos demográficos de OWID (2000–2022)
# Nombre: Abraham Cano
**Objetivo:** descargar cuatro conjuntos de datos, revisar sus columnas, preparar cada uno por separado y unirlos por `Entity` y `Year`. El resultado servirá para una práctica posterior de aprendizaje automático.

Ejecuta las celdas en orden. Requiere `pandas` y `matplotlib`, además de acceso a internet. Fuente: [Our World in Data](https://ourworldindata.org/).

In [0]:
import pandas as pd
import matplotlib.pyplot as plt
import requests

OPCIONES_OWID = {"User-Agent": "Our World In Data data fetch/1.0"}
BASE = "https://ourworldindata.org/grapher/"
SUFIJO = ".csv?v=1&csvType=full&useColumnShortNames=true"
from pathlib import Path
import json

# Usar las descargas incluidas; descargar únicamente si falta un archivo.
def leer_owid(slug):
    local = Path("../data/raw/owid") / f"{slug}.csv"
    if local.exists():
        return pd.read_csv(local)
    return pd.read_csv(BASE + slug + SUFIJO, storage_options=OPCIONES_OWID)


## 1. Descargar los cuatro archivos

Las direcciones son cadenas de texto simples. Cada archivo se carga en su propio DataFrame.

In [0]:
url_population = BASE + "population-with-un-projections" + SUFIJO
url_births = BASE + "number-of-births-per-year" + SUFIJO
url_deaths = BASE + "number-of-deaths-per-year" + SUFIJO
url_child_deaths = BASE + "number-of-child-deaths-unwpp" + SUFIJO 
url_GDP_Healt = BASE + "total-healthcare-expenditure-gdp" + SUFIJO
url_gdp = BASE + "gdp-per-capita-worldbank" + SUFIJO

population = leer_owid("population-with-un-projections")
births = leer_owid("number-of-births-per-year")
deaths = leer_owid("number-of-deaths-per-year")
child_deaths = leer_owid("number-of-child-deaths-unwpp")
HEALTH = leer_owid("total-healthcare-expenditure-gdp")
gdp_df = leer_owid("gdp-per-capita-worldbank")




print("Filas y columnas:")
print("population:", population.shape)
print("births:", births.shape)
print("deaths:", deaths.shape)
print("child_deaths:", child_deaths.shape)
print("GDP_Health", HEALTH.shape)
print("Filas y columnas de GDP:", gdp_df.shape)
vida_df = leer_owid("life-expectancy-at-birth-including-the-un-projections")


In [0]:
HEALTH.head(3)

In [0]:
gdp_df.head(3)

In [0]:
ruta_metadata = Path("../data/raw/owid/total-healthcare-expenditure-gdp.metadata.json")
if ruta_metadata.exists():
    metadata = json.loads(ruta_metadata.read_text())
else:
    respuesta = requests.get(BASE + "total-healthcare-expenditure-gdp.metadata.json", timeout=30)
    respuesta.raise_for_status()
    metadata = respuesta.json()
print(metadata["columns"])


## 2. Explorar las columnas

OWID puede modificar los nombres de los indicadores. Revisamos las columnas reales antes de seleccionarlas.

In [0]:
for nombre, tabla in [("population", population), 
                      ("births", births),
                      ("deaths", deaths), 
                      ("child_deaths", child_deaths)]:
    print(f"\n{nombre}: {tabla.columns.tolist()}")
    display(tabla.head(3))

## 3. Preparar población

Elegimos la columna de **estimaciones históricas** para ambos sexos y todas las edades; excluimos las proyecciones. Después conservamos los años de interés.

In [0]:
columnas_population = [
    columna for columna in population.columns
        if "population" in columna and "estimates" in columna]

print("Columnas candidatas:", columnas_population)

assert len(columnas_population) == 1, "Revisar manualmente el indicador de población mostrado arriba"


col_population = columnas_population[0]

population_limpio = population[["entity", "year", col_population]].copy()

population_limpio = population_limpio.rename(columns={col_population: "population"})

population_limpio = population_limpio[population_limpio["year"].between(2000, 2022)].copy()

print(population_limpio.shape)
display(population_limpio.head())

## 4. Preparar nacimientos

Seleccionamos el número anual de nacimientos históricos. Revisamos qué entidades comparte con las otras tablas.

In [0]:
columnas_births = [c for c in births.columns
                  if "births" in c and "estimates" in c]


print("Columnas candidatas:", columnas_births)
assert len(columnas_births) == 1, "Revisar manualmente el indicador de nacimientos mostrado arriba"
col_births = columnas_births[0]

births_limpio = births[["entity", "year", col_births]].copy()
births_limpio = births_limpio.rename(columns={col_births: "births"})
births_limpio = births_limpio[births_limpio["year"].between(2000, 2022)].copy()
print(births_limpio.shape)
display(births_limpio.head())

## 5. Preparar defunciones totales

In [0]:
columnas_deaths = [c for c in deaths.columns
                   if "deaths" in c and "estimates" in c]

print("Columnas candidatas:", columnas_deaths)
assert len(columnas_deaths) == 1, "Revisar manualmente el indicador de defunciones mostrado arriba"

col_deaths = columnas_deaths[0]

deaths_limpio = deaths[["entity", "year", col_deaths]].copy()
deaths_limpio = deaths_limpio.rename(columns={col_deaths: "deaths"})
deaths_limpio = deaths_limpio[deaths_limpio["year"].between(2000, 2022)].copy()

print(deaths_limpio.shape)
display(deaths_limpio.head())

## 6. Preparar defunciones de menores de cinco años

Este indicador corresponde a niños de 0 a 4 años; no debe confundirse con mortalidad de menores de un año.

In [0]:
columnas_child = [c for c in child_deaths.columns
                  if c not in ["entity", "code", "year"]]

print("Columnas candidatas:", columnas_child)

assert len(columnas_child) == 1, "Revisar manualmente el indicador de menores de cinco años mostrado arriba"

col_child = columnas_child[0]

child_deaths_limpio = child_deaths[["entity", "year", col_child]].copy()
child_deaths_limpio = child_deaths_limpio.rename(columns={col_child: "child_deaths_under_5"})

child_deaths_limpio = child_deaths_limpio[child_deaths_limpio["year"].between(2000, 2022)].copy()

print(child_deaths_limpio.shape)
display(child_deaths_limpio.head())

In [0]:
col_health = [c for c in HEALTH.columns if "current_health_expenditure" in c][0]
health_limpio = HEALTH[["entity", "year", col_health]].copy()
health_limpio = health_limpio.rename(columns={col_health: "percentage_health"})
health_limpio = health_limpio[health_limpio["year"].between(2000, 2022)].copy()

col_gdp = "ny_gdp_pcap_pp_kd"
gdp_limpio = gdp_df[["entity", "year", col_gdp]].copy()
gdp_limpio = gdp_limpio.rename(columns={col_gdp: "gdp"})
gdp_limpio = gdp_limpio[gdp_limpio["year"].between(2000, 2022)].copy()

print("Columnas HEALTH limpio:", health_limpio.columns.tolist())
print("Columnas GDP limpio:", gdp_limpio.columns.tolist())
col_vida = "life_expectancy__sex_all__age_0__variant_estimates"
vida_limpio = vida_df.loc[vida_df["year"].between(2000, 2022),
                         ["entity", "code", "year", col_vida]].copy()
vida_limpio = vida_limpio.rename(columns={col_vida: "life_expectancy"})


## 7. Revisar llaves y cobertura

Cada combinación `Entity`–`Year` debe aparecer una sola vez en cada tabla. `Code` no se utiliza como llave porque puede faltar en agregados regionales.

In [0]:
tablas = [("population", population_limpio), ("births", births_limpio),
          ("deaths", deaths_limpio), ("child_deaths", child_deaths_limpio)]

for nombre, tabla in tablas:
    duplicados = tabla.duplicated(["entity", "year"]).sum()
    print(nombre, "| entidades:", tabla["entity"].nunique(),
          "| años:", tabla["year"].min(), "a", tabla["year"].max(),
          "| llaves duplicadas:", duplicados)
    assert duplicados == 0, f"Llaves duplicadas en {nombre}"

entidades_comunes = set(population_limpio["entity"]) & set(births_limpio["entity"]) \
                  & set(deaths_limpio["entity"]) & set(child_deaths_limpio["entity"])

print("Entidades presentes en los cuatro archivos:", len(entidades_comunes))
print(sorted(entidades_comunes)[:30])

## 8. Unir las tablas, un paso a la vez

Usamos `inner`: permanecen únicamente las combinaciones de entidad y año presentes en ambos DataFrames. `validate="one_to_one"` detecta llaves duplicadas.

In [0]:
population_births = pd.merge(
    population_limpio, births_limpio,
    on=["entity", "year"], how="inner", validate="one_to_one"
)
print("Después de población + nacimientos:", population_births.shape)
display(population_births.head())

In [0]:
population_births_deaths = pd.merge(
    population_births, deaths_limpio,
    on=["entity", "year"], how="inner", validate="one_to_one"
)
print("Después de agregar defunciones:", population_births_deaths.shape)
display(population_births_deaths.head())

In [0]:
demografia = pd.merge(
    population_births_deaths, child_deaths_limpio,
    on=["entity", "year"], how="inner", validate="one_to_one"
)
demografia = pd.merge(
    demografia, health_limpio,
    on=["entity", "year"], how="inner", validate="one_to_one"
)

demografia = pd.merge(
    demografia, gdp_limpio,
    on=["entity", "year"], how="inner", validate="one_to_one"
)

demografia = demografia.merge(vida_limpio, on=["entity", "year"], how="inner", validate="one_to_one")
print("Dataset integrado:", demografia.shape)
display(demografia.head())

## 9. Revisar faltantes y crear tasas

Eliminamos solo filas con algún indicador necesario ausente. Las tasas de nacimientos y defunciones se expresan por 1,000 habitantes. La razón de muertes de menores de cinco años usa nacimientos del **mismo año** como denominador; es una medida descriptiva aproximada, no una tasa de mortalidad de cohorte.

In [0]:
print("Valores faltantes antes de limpiar:")
print(demografia.isna().sum())
print("Filas antes de limpiar:", len(demografia))

demografia = demografia.dropna(subset=["population", "births", "deaths", "child_deaths_under_5", "percentage_health", "gdp", "life_expectancy"]).copy()
demografia = demografia[(demografia["population"] > 0) & (demografia["births"] > 0)].copy()

print("Filas después de limpiar:", len(demografia))
print("Entidades después de limpiar:", demografia["entity"].nunique())
assert not demografia.empty, "No hay entidades y años coincidentes; revisar fuentes y columnas"

In [0]:
demografia["birth_rate_per_1000"] = demografia["births"] / demografia["population"] * 1000
demografia["death_rate_per_1000"] = demografia["deaths"] / demografia["population"] * 1000
demografia["under5_deaths_per_1000_births_same_year"] = (
    demografia["child_deaths_under_5"] / demografia["births"] * 1000
)

demografia = demografia[[
    "entity", "code", "year", "population", "births", "deaths", "child_deaths_under_5",
    "percentage_health", "gdp", "life_expectancy",
    "birth_rate_per_1000", "death_rate_per_1000",
    "under5_deaths_per_1000_births_same_year"
]].sort_values(["entity", "year"]).reset_index(drop=True)

display(demografia.head())
display(demografia.describe(include="all"))

## 10. Visualizar una entidad disponible

Puedes cambiar `entidad` por cualquier nombre de `demografia["Entity"].unique()`.

In [0]:
entidad = "Mexico" if "Mexico" in demografia["entity"].values else demografia["entity"].iloc[0]
ejemplo = demografia[demografia["entity"] == entidad].sort_values("year")

ejemplo.plot(x="year", y=["birth_rate_per_1000", "death_rate_per_1000"],
             marker="o", figsize=(9, 4), title=f"Tasas demográficas: {entidad}")
plt.ylabel("Por 1,000 habitantes")
plt.grid(alpha=0.3)
plt.show()

## 11. Guardar el resultado

El CSV se guarda junto al notebook. Contiene solamente entidades y años compartidos por las cuatro fuentes.

In [0]:
# Guardar el df completo
 
from pathlib import Path
 
carpeta_salida = Path("../data/raw")
carpeta_salida.mkdir(parents=True, exist_ok=True)
 
salida = carpeta_salida / "dataset_7variables_2000_2022.csv"
 
columnas_modelo = ["entity", "code", "year", "population", "births", "deaths",
                   "child_deaths_under_5", "percentage_health", "gdp", "life_expectancy"]
dataset_modelo = demografia[columnas_modelo].copy()
assert not dataset_modelo.duplicated(["entity", "year"]).any()
assert dataset_modelo[columnas_modelo[3:]].notna().all().all()
dataset_modelo.to_csv(salida, index=False)
 
print(f"Archivo guardado: {salida.resolve()} \n | filas: {len(demografia)}")